
# Web Crawler System Design

## What is a Web Crawler?

A Web Crawler is a system that automatically visits web pages, downloads content, extracts links, and recursively crawls websites.

Also called:
- Spider
- Bot
- Scraper

Used by:
- Google Search
- Bing
- SEO tools

---

# Functional Requirements

- Crawl web pages
- Extract links
- Avoid duplicate crawling
- Respect robots.txt
- Store crawled content

---

# Non Functional Requirements

- Scalability
- Fault tolerance
- High throughput
- Distributed crawling

---

# High Level Architecture (HLD)

```text
Seed URLs
    ↓
URL Frontier
    ↓
Crawler Workers
    ↓
Fetcher
    ↓
Parser
    ↓
Deduplication
    ↓
Storage / Search Index
```

---

# URL Frontier

Stores URLs waiting to crawl.

Acts like:
- Distributed queue

Supports:
- prioritization
- retries
- deduplication

---

# Crawl Workflow

```text
1. Get URL from frontier
2. Download webpage
3. Parse HTML
4. Extract links
5. Add new URLs
6. Store page content
7. Repeat
```

---

# Page Fetcher

Responsible for:
- HTTP requests
- retries
- redirects
- compression
- timeouts

Optimizations:
- DNS caching
- connection pooling
- async IO

---

# HTML Parser

Extract:
- title
- metadata
- text
- links

Example:

```html
<a href="https://google.com">
```

Extracted URL:

```text
https://google.com
```

---

# URL Normalization

These may be same:

```text
https://google.com
https://google.com/
https://GOOGLE.com
```

Need normalization before storage.

---

# Duplicate Detection

Avoid:
- duplicate crawling
- infinite loops

Techniques:
- hashing
- Bloom Filters

---

# Bloom Filters

Advantages:
- memory efficient
- O(1) lookup
- scalable

---

# robots.txt Handling

Crawler must respect:

```text
robots.txt
```

Example:

```text
Disallow: /admin
```

---

# Politeness Policy

Avoid overloading websites.

Rules:
- crawl delay
- rate limiting
- per-domain concurrency limits

Example:

```text
1 request/sec/domain
```

---

# Distributed Crawling

Architecture:

```text
URL Frontier
      ↓
Distributed Crawl Workers
      ↓
Distributed Storage
```

Challenges:
- duplicate crawling
- coordination
- balancing

Solutions:
- Kafka queues
- distributed locks
- consistent hashing

---

# URL Partitioning

Use:

```text
hash(domain) % N
```

Example:

```text
google.com → Worker 1
amazon.com → Worker 2
```

---

# Scheduler

Determines:
- crawl priority
- refresh frequency

Factors:
- popularity
- freshness
- importance

---

# Incremental Crawling

Use:
- ETag
- Last-Modified headers

Example:

```http
304 Not Modified
```

Saves bandwidth.

---

# Storage Design

Store:
- raw HTML
- metadata
- crawl status

Storage choices:

| Data | Storage |
|---|---|
| HTML | Object Storage |
| Metadata | SQL/NoSQL |
| Search Index | Elasticsearch |

---

# Search Indexing

Pipeline:

```text
Crawler
   ↓
Parser
   ↓
Indexer
   ↓
Search Engine
```

---

# Failure Handling

## Worker Crash
- retries
- checkpointing

## Retry Strategy

```text
1 sec
2 sec
4 sec
8 sec
```

---

# Infinite URL Problem

Example:

```text
?page=1
?page=2
?page=999999
```

Solutions:
- URL filtering
- max depth
- canonical URLs

---

# JavaScript Rendering

Need:
- Headless Chrome
- Puppeteer
- Playwright

Tradeoff:
- slower
- expensive

---

# Security

Crawler may encounter:
- malware
- phishing
- huge payloads

Need:
- sandboxing
- validation
- limits

---

# Monitoring

Track:
- crawl rate
- failed URLs
- queue size
- storage growth

Tools:
- Prometheus
- Grafana

---

# Low Level Design (LLD)

Main Classes:

```text
CrawlerWorker
URLFrontier
Fetcher
Parser
Deduplicator
Scheduler
StorageService
```

---

# URL Entity

```java
class URLInfo {

    String url;
    int priority;
    int retryCount;
    Date lastCrawled;
    boolean visited;
}
```

---

# Fetcher

```java
class Fetcher {

    public String fetch(String url) {

        return html;
    }
}
```

---

# Parser

```java
class Parser {

    public List<String> extractLinks(String html) {

        return urls;
    }
}
```

---

# Deduplicator

```java
class Deduplicator {

    BloomFilter bloomFilter;

    public boolean isDuplicate(String url) {

        return bloomFilter.contains(url);
    }
}
```

---

# Worker Flow

```java
class CrawlerWorker {

    public void crawl() {

        String url = frontier.getNext();

        String html = fetcher.fetch(url);

        List<String> links = parser.extractLinks(html);

        frontier.add(links);

        storage.store(html);
    }
}
```

---

# Final Production Architecture

```text
Seed URLs
    ↓
Kafka Frontier
    ↓
Crawler Workers
    ↓
JS Renderers
    ↓
Distributed DB
    ↓
Search Index
```

---

# Final Interview Summary

A scalable web crawler uses:
- distributed URL frontier
- async crawl workers
- Bloom filters for deduplication
- distributed storage
- schedulers for prioritization
- incremental crawling
- Kafka queues for scalability
